In [1]:
import ast
import os
import pandas as pd

In [2]:
class CleanAndNormalizeAST(ast.NodeTransformer):
    def __init__(self):
        self.id_map = {}
        self.counter = 0

    def visit_Expr(self, node):
        if isinstance(node.value, ast.Constant) and isinstance(node.value.value, str):
            return None 
        return self.generic_visit(node)

    def visit_Name(self, node):
        if isinstance(node.ctx, (ast.Store, ast.Load)):
            if node.id not in self.id_map:
                self.counter += 1
                self.id_map[node.id] = f"var_{self.counter}"
            node.id = self.id_map[node.id]
        return self.generic_visit(node)

    def visit_FunctionDef(self, node):
        if node.name not in self.id_map:
            self.counter += 1
            self.id_map[node.name] = f"func_{self.counter}"
        node.name = self.id_map[node.name]
        return self.generic_visit(node)

In [3]:
def process_file_to_ast(file_path):
    try:
        with open(file_path, 'r', encoding='utf-8') as f:
            source_code = f.read()
        raw_tree = ast.parse(source_code)
        
        transformer = CleanAndNormalizeAST()
        clean_tree = transformer.visit(raw_tree)
 
        ast.fix_missing_locations(clean_tree)
        
        return ast.dump(clean_tree, annotate_fields=False)
    except Exception as e:
        print(f"   [!] Error parsing {os.path.basename(file_path)}: {e}")
        return None

In [4]:
def build_final_dataset():
    print("[*] Loading labeled dataset...")
    df = pd.read_csv("Data/cheating_dataset.csv")
    
    ast_cache = {}
    def get_cached_ast(file_name):
        if file_name not in ast_cache:
            full_path = os.path.join("Data/original-codes/", file_name)
            ast_cache[file_name] = process_file_to_ast(full_path)
        return ast_cache[file_name]
        
    print("[*] Parsing raw code into anonymized AST strings...")
    df['AST_1'] = df['File_1'].apply(get_cached_ast)
    df['AST_2'] = df['File_2'].apply(get_cached_ast)
    
    initial_len = len(df)
    df_cleaned = df.dropna(subset=['AST_1', 'AST_2']).reset_index(drop=True)
    
    df_cleaned.to_csv("Data/ast_dataset.csv", index=False)
    print(f"[+] Success! {len(df_cleaned)}/{initial_len} pairs saved to Data/ast_dataset.csv.")

In [5]:
if __name__ == "__main__":
    build_final_dataset()

[*] Loading labeled dataset...
[*] Parsing raw code into anonymized AST strings...
[+] Success! 293/293 pairs saved to Data/ast_dataset.csv.
